In [1]:
pip install bitsandbytes accelerate peft trl datasets sentencepiece protobuf huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 30.3 MB/s eta 0:00:00


In [2]:
!pip install sentencepiece protobuf datasets huggingface_hub hf_transfer
!pip install --upgrade unsloth

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 92.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.4/82.4 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.4/28.4 MB 80.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 122.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 72.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 109.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 105.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 113.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.5/230.5 kB 28.7 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully u

In [3]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3.1-8B",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.10.3: Fast Llama patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Meta-Llama-3.1-8B-unsloth-bnb-4bit as a legacy tokenizer.


In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

Unsloth 2026.10.3 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [28]:
from datasets import load_dataset

dataset = load_dataset("xlangai/spider")

# Load the existing training and validation splits
train_dataset = dataset["train"]
test_dataset = dataset["validation"]

split_dataset = dataset["train"].train_test_split(
    test_size=0.1,
    seed=42
)

train_dataset = split_dataset["train"]
validation_dataset = split_dataset["test"]

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

Train: 6300
Validation: 700
Test: 1034


In [20]:
# Access individual examples
print(dataset["train"][0])

# For fine-tuning
train_dataset = dataset["train"]


{'db_id': 'department_management', 'query': 'SELECT count(*) FROM head WHERE age  >  56', 'question': 'How many heads of the departments are older than 56 ?', 'query_toks': ['SELECT', 'count', '(', '*', ')', 'FROM', 'head', 'WHERE', 'age', '>', '56'], 'query_toks_no_value': ['select', 'count', '(', '*', ')', 'from', 'head', 'where', 'age', '>', 'value'], 'question_toks': ['How', 'many', 'heads', 'of', 'the', 'departments', 'are', 'older', 'than', '56', '?'], 'text': 'Below is an instruction that describes a task, paired with an input that provides further context.\n\n### Instruction:\nGenerate the SQL query first, then explain how it works.\n\n### Input:\nHow many heads of the departments are older than 56 ?\n\n### Response:\nSQL Query:\nSELECT count(*) FROM head WHERE age  >  56\n\nExplanation:\nThis SQL query answers the given question by retrieving the relevant data from the database. Its exact operation depends on the SQL clauses, tables, and conditions used in the query.<|end_of_t

In [51]:

import os
import urllib.request
import zipfile

# Download the official Spider GitHub repository
url = "https://github.com/taoyds/spider/archive/refs/heads/master.zip"
zip_path = "/content/spider-master.zip"

urllib.request.urlretrieve(url, zip_path)

# Extract the ZIP file
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall("/content/")

# Locate tables.json
tables_path = "/content/spider-master/evaluation_examples/examples/tables.json"

print("ZIP downloaded:", os.path.exists(zip_path))
print("tables.json found:", os.path.exists(tables_path))
print("ZIP location:", zip_path)


ZIP downloaded: True
tables.json found: True
ZIP location: /content/spider-master.zip


In [52]:
import json

with open(tables_path, "r", encoding="utf-8") as f:
    schemas = json.load(f)

schema_dict = {item["db_id"]: item for item in schemas}

print("Number of database schemas:", len(schema_dict))
print("Example database IDs:", list(schema_dict.keys())[:5])

Number of database schemas: 166
Example database IDs: ['perpetrator', 'college_2', 'flight_company', 'icfp_1', 'body_builder']


In [53]:
def get_schema(db_id):
    db = schema_dict[db_id]

    tables = db["table_names"]
    columns = db["column_names"]

    schema_text = []

    for table_id, table_name in enumerate(tables):
        column_names = [
            column_name
            for column_table_id, column_name in columns
            if column_table_id == table_id
        ]

        schema_text.append(
            f"{table_name}({', '.join(column_names)})"
        )

    return "\n".join(schema_text)

In [54]:
print(get_schema("department_management"))

department(department id, name, creation, ranking, budget in billions, num employees)
head(head id, name, born state, age)
management(department id, head id, temporary acting)


In [55]:
alpaca_prompt = """### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    texts = []


    for question, query, db_id in zip(
        examples["question"],
        examples["query"],
        examples["db_id"]
    ):
        schema = get_schema(db_id)  # Implement using Spider's tables.json

        text = alpaca_prompt.format(
            schema,
            question,
            query
        ) + tokenizer.eos_token

        texts.append(text)

    return {"text": texts}


dataset = dataset.map(
    formatting_prompts_func,
    batched=True
)

Map:   0%|          | 0/7000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1034 [00:00<?, ? examples/s]

In [56]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

# Select the training split
trained_mapped_dataset = dataset["train"]

# Verify that formatting has already been applied
print(trained_mapped_dataset.column_names)
print(trained_mapped_dataset[0]["text"][:500])

def formatting_func(example):
    return example["text"]

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=trained_mapped_dataset,
    formatting_func=formatting_func,
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)

trainer.train()

['db_id', 'query', 'question', 'query_toks', 'query_toks_no_value', 'question_toks', 'text']
### Instruction:
department(department id, name, creation, ranking, budget in billions, num employees)
head(head id, name, born state, age)
management(department id, head id, temporary acting)

### Input:
How many heads of the departments are older than 56 ?

### Response:
SELECT count(*) FROM head WHERE age  >  56<|end_of_text|>
Unsloth: transformers renamed `push_to_hub_token` to `hub_token`. Forwarding your value to `hub_token` - update your code when convenient. If you also passed `hub_token` as None, that is its default here and cannot be distinguished from leaving it unset, so `push_to_hub_token` was used; drop `push_to_hub_token` to keep it.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/7000 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 7,000 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,1.873210
2,1.807947
3,1.565519
4,1.273895
5,1.274337
6,1.325123
7,1.134858
8,1.321670
9,1.037414
10,0.970297


TrainOutput(global_step=60, training_loss=0.8636919577916463, metrics={'train_runtime': 426.5717, 'train_samples_per_second': 1.125, 'train_steps_per_second': 0.141, 'total_flos': 4383401891954688.0, 'train_loss': 0.8636919577916463, 'epoch': 0.06857142857142857})

In [58]:
model.save_pretrained("spider-sql-model")
tokenizer.save_pretrained("spider-sql-model")

('spider-sql-model/tokenizer_config.json', 'spider-sql-model/tokenizer.json')

In [64]:
schema = """
department(department_id, name, building, budget)
"""
question = "Find department names with a budget greater than 50000."

prompt = alpaca_prompt.format(
    "Generate the SQL query first, then explain each part of the query and what output it returns.",

                              schema,
    question,
    ""

)

inputs = tokenizer(
    [prompt],
    return_tensors="pt"
).to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    do_sample=False,
    use_cache=True,
    pad_token_id=tokenizer.eos_token_id,
)

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)

Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


SELECT name FROM department WHERE budget  >  50000


In [73]:
!ls /content

Fine-tune-text-sql	      sample_data	 spider-sql-model
huggingface_tokenizers_cache  spider-master	 unsloth_compiled_cache
outputs			      spider-master.zip


In [67]:
!git config --global user.name "gaurav77859"
!git config --global user.email "gkk116180@gmail.com "

In [71]:
!git clone  https://github.com/gaurav77859/Fine-tune-text-sql.git

Cloning into 'Fine-tune-text-sql'...


In [72]:
%cd Fine-tune-text-sql.git
!git add .
!git commit -m "Add text-to-SQL Alpaca notebook"
!git push

[Errno 2] No such file or directory: 'Fine-tune-text-sql.git'
/content
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
